# Notebook 3: Silver Layer - Nettoyage et Standardisation

**Objectif**: Transformer les données brutes de la couche Bronze en données propres et standardisées.

## Qu'est-ce que la couche Silver?

La **couche Silver** est la seconde couche du lakehouse:
- **Nettoyage**: Suppression des données invalides
- **Standardisation**: Noms de colonnes cohérents (snake_case)
- **Typage**: Conversion des types de données appropriés
- **Déduplication**: Suppression des doublons

## Flux de données

```
Bronze (raw) ──► Filtrage ──► Standardisation ──► Silver (clean)
```

---
## INITIALISATION

In [ ]:
# Configuration du chemin vers le projet
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import des modules Spark et configuration
from lakehouse.spark_session import get_spark

# Création de la session Spark
spark = get_spark("silver-transformations")

# Configuration de la branche Nessie main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Création des namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

print("=" * 60)
print("COUCHE SILVER - NETTOYAGE ET STANDARDISATION")
print("=" * 60)

## 1. Lecture de la table Bronze

La couche Silver lit depuis Bronze, jamais depuis la source originale.

In [ ]:
# Lecture de la table Bronze
sales_bronze = spark.table("nessie.bronze.sales")

bronze_count = sales_bronze.count()
print(f"Enregistrements dans Bronze: {bronze_count:,}")

# Aperçu des données
print("\n=== Aperçu Bronze ===")
sales_bronze.select("Order ID", "Order Date", "Customer Name", "Sales", "Quantity").show(5, truncate=False)

## 2. Standardisation des noms de colonnes

Conversion du format **PascalCase** (Bronze) vers **snake_case** (Silver):

| Bronze | Silver |
|--------|--------|
| `Order ID` | `order_id` |
| `Order Date` | `order_date` |
| `Customer Name` | `customer_name` |
| `Sales` | `sales` |

In [ ]:
from pyspark.sql.functions import col

# Renommage des colonnes et standardisation
sales_silver = (
    sales_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        # Métadonnées préservées
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
)

print("=== Données standardisées ===")
sales_silver.select("order_id", "order_date", "customer_name", "sales", "quantity").show(5, truncate=False)

## 3. Qualité des données - Filtrage

Suppression des enregistrements invalides:
- Valeurs NULL sur les champs critiques (order_id, product_id, sales)
- Doublons (même order_id + product_id)

In [ ]:
# Filtrage des données invalides
sales_silver_clean = (
    sales_silver
    # Supprimer les NULL sur les champs critiques
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    # Déduplication sur la clé naturelle
    .dropDuplicates(["order_id", "product_id"])
)

silver_count = sales_silver_clean.count()
filtered_count = bronze_count - silver_count

print(f"Bronze: {bronze_count:,} enregistrements")
print(f"Silver: {silver_count:,} enregistrements")
print(f"Filtrés: {filtered_count:,} enregistrements ({filtered_count/bronze_count*100:.2f}%)")

## 4. Vérification de la qualité des données

Contrôles pour s'assurer que les données Silver sont propres.

In [ ]:
from pyspark.sql.functions import sum, when, count as spark_count

# Vérifier les NULL
null_checks = sales_silver_clean.select(
    spark_count(when(col("order_id").isNull(), 1)).alias("null_order_id"),
    spark_count(when(col("product_id").isNull(), 1)).alias("null_product_id"),
    spark_count(when(col("sales").isNull(), 1)).alias("null_sales"),
)

print("=== Contrôles de qualité ===")
print("Valeurs NULL:")
null_checks.show()

# Vérifier les valeurs négatives
neg_sales = sales_silver_clean.filter(col("sales") < 0).count()
neg_quantity = sales_silver_clean.filter(col("quantity") <= 0).count()
invalid_discount = sales_silver_clean.filter((col("discount") < 0) | (col("discount") > 1)).count()

print(f"Ventes négatives: {neg_sales}")
print(f"Quantités invalides (<= 0): {neg_quantity}")
print(f"Remises invalides: {invalid_discount}")

## 5. Suppression de l'ancienne table Silver (si elle existe)

In [ ]:
# Nettoyage: suppression de la table si elle existe
spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")

print("Tables existantes supprimées")

## 6. Création de la table Silver

In [ ]:
# Création de la table Silver
sales_silver_clean.writeTo("nessie.silver.sales").using("iceberg").create()

print("Table Silver créée: nessie.silver.sales")

## 7. Vérification et exemples de requêtes

In [ ]:
# Vérifier le count
final_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.sales").first()[0]
print(f"Enregistrements dans Silver: {final_count:,}")

In [ ]:
# Exemple d'analyse: Ventes par catégorie
print("=== Ventes par catégorie ===")
spark.sql("""
    SELECT 
        category,
        ROUND(SUM(sales), 2) as total_sales,
        SUM(quantity) as total_quantity,
        COUNT(DISTINCT order_id) as order_count
    FROM nessie.silver.sales
    GROUP BY category
    ORDER BY total_sales DESC
""").show(truncate=False)

In [ ]:
# Distribution par batch
print("=== Distribution par batch ===")
spark.sql("""
    SELECT 
        batch_id,
        COUNT(*) as count,
        ROUND(SUM(sales), 2) as total_sales
    FROM nessie.silver.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show(truncate=False)

In [ ]:
# Top 5 des produits
print("=== Top 5 des produits ===")
spark.sql("""
    SELECT 
        category,
        product_name,
        ROUND(SUM(sales), 2) as total_sales,
        SUM(quantity) as total_quantity
    FROM nessie.silver.sales
    GROUP BY category, product_name
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

## RÉSUMÉ DE LA COUCHE SILVER

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         COUCHE SILVER - TRANSFORMATION TERMINÉE                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Table               nessie.silver.sales                                 ║
║  Format              Iceberg                                             ║
║                                                                          ║
║  Transformations:                                                        ║
║    - Colonnes renommées (snake_case)                                     ║
║    - Types convertis (double, int)                                       ║
║    - NULL filtrés sur champs critiques                                   ║
║    - Doublons supprimés                                                  ║
║                                                                          ║
║  Volumes:                                                                ║
║    Bronze: {:>10,} enregistrements                                       ║
║    Silver: {:>10,} enregistrements                                       ║
║    Filtrés: {:>10,} ({:>5.1f}%)                                          ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""".format(bronze_count, final_count, filtered_count, filtered_count/bronze_count*100))

print("→ Prochaine étape: Exécuter '04_gold_layer.ipynb'")